#Time Series Analysis: Nigeria CPI and Nigeria-USD Exchange Rate.

**Course:** NUTDTS816 - Time Series Analysis

**Series A:** Nigeria Consumer Price Index(All Items)

**Series B:** Nigeria-USD Exchange Rate

**Data Source:** NUTDTS816 Github Course Repository

#Purpose

This notebook explores, visualises and decomposes two monthly Nigerian economic time series before forecasting. Series A is the Nigeria Consumer Price Index(CPI), while Series B is the Nigeria-USD exchange rate. The analysis examines trend, seasonality, serial dependence, decomposition, transformation and seasonal adjustment, with particluar attention to features that may affect subsequent forecasting.

In [35]:
#Core Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

#Statistical/Time-series tools
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.tsa.seasonal import seasonal_decompose, STL
from pandas.plotting import lag_plot

#Display settings
pd.set_option("display.max_columns", None)

#plot style
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.grid"] = True

In [36]:
#Raw GitHub URLs
CPI_URL = "https://raw.githubusercontent.com/toadesina/NUTDTS816_Course_Repository/main/data/nigeria_cpi.csv"

FX_URL = "https://raw.githubusercontent.com/toadesina/NUTDTS816_Course_Repository/main/data/nigeria_fx.csv"

In [37]:
#Load the Datasets
cpi_raw = pd.read_csv(CPI_URL)
fx_raw = pd.read_csv(FX_URL)

print("CPI shape:", cpi_raw.shape)
print("FX shape:", fx_raw.shape)

CPI shape: (144, 2)
FX shape: (139, 2)


In [38]:
#Data Inspection
print("CPI DATA")
display(cpi_raw.head())

print("\nFX DATA")
display(fx_raw.head())

print("CPI information")
cpi_raw.info()

print("\nFX information")
fx_raw.info()

print("CPI descriptive statistics")
display(cpi_raw.describe())

print("\nFX descriptive statistics")
display(fx_raw.describe())

print("CPI columns:",
cpi_raw.columns.tolist())
print("FX columns:", fx_raw.columns.tolist())

CPI DATA


,date,cpi
0,2014-01-01,20.1179
1,2014-02-01,20.2179
2,2014-03-01,20.3766
3,2014-04-01,20.5019
4,2014-05-01,20.6615



FX DATA


,date,ngn_usd
0,2015-01-01,167.5000
1,2015-02-01,178.1500
2,2015-03-01,196.5727
3,2015-04-01,196.5000
4,2015-05-01,196.5000


CPI information
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 144 entries, 0 to 143
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   date    144 non-null    object 
 1   cpi     144 non-null    float64
dtypes: float64(1), object(1)
memory usage: 2.4+ KB

FX information
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 139 entries, 0 to 138
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   date     139 non-null    object 
 1   ngn_usd  139 non-null    float64
dtypes: float64(1), object(1)
memory usage: 2.3+ KB
CPI descriptive statistics


,cpi
count,144.000000
mean,52.109399
std,31.250618
min,20.117900
25%,28.245175
50%,40.536050
75%,65.853225
max,131.166600



FX descriptive statistics


,ngn_usd
count,139.000000
mean,602.831804
std,485.869077
min,167.500000
25%,305.366500
50%,379.500000
75%,765.718400
max,1670.276400


CPI columns: ['date', 'cpi']
FX columns: ['date', 'ngn_usd']


#Task 1 - Load and Tidy the Series

In [39]:
#Convert date columns to datetime
cpi_raw["date"] = pd.to_datetime(cpi_raw["date"])
fx_raw["date"] = pd.to_datetime(fx_raw["date"])

#Check the resulting data types
print(cpi_raw.dtypes)
print()
print(fx_raw.dtypes)

date    datetime64[ns]
cpi            float64
dtype: object

date       datetime64[ns]
ngn_usd           float64
dtype: object


In [40]:
#Checking Chronological ordering
print("CPI dates sorted:", cpi_raw["date"].is_monotonic_increasing)
print("FX dates sorted:", fx_raw["date"].is_monotonic_increasing)

cpi_raw = cpi_raw.sort_values("date")
fx_raw = fx_raw.sort_values("date")

CPI dates sorted: True
FX dates sorted: True


In [41]:
#Checking Duplicate Timestamps
print("CPI duplicate timestamps:", cpi_raw["date"].duplicated().sum())
print("FX duplicate timestamps:", fx_raw["date"].duplicated().sum())

CPI duplicate timestamps: 0
FX duplicate timestamps: 0


No duplicated timestamps were detected in either series; therefore, no observations were removed on the basis of timestamp duplication.

In [42]:
#Creating the datetime index
cpi = cpi_raw.set_index("date").copy()

fx = fx_raw.set_index("date").copy()

cpi.head()

fx.head()

,ngn_usd
date,
2015-01-01,167.5000
2015-02-01,178.1500
2015-03-01,196.5727
2015-04-01,196.5000
2015-05-01,196.5000


In [43]:
#Setting the frequency
cpi = cpi.asfreq("MS")
fx = fx.asfreq("MS")

print("CPI frequency:", cpi.index.freq)
print("FX frequency:", fx.index.freq)

CPI frequency: <MonthBegin>
FX frequency: <MonthBegin>


In [44]:
#Checking for missing timestamps
cpi_missing_dates = pd.date_range(start=cpi.index.min(), end=cpi.index.max(), freq="MS").difference(cpi.index)

fx_missing_dates = pd.date_range(start=fx.index.min(), end=fx.index.max(), freq="MS").difference(fx.index)

print("Missing CPI timestamps:")
print(cpi_missing_dates)

print("\nMissing FX timestamps:")
print(fx_missing_dates)

Missing CPI timestamps:
DatetimeIndex([], dtype='datetime64[ns]', freq='MS')

Missing FX timestamps:
DatetimeIndex([], dtype='datetime64[ns]', freq='MS')


In [45]:
#Checking for missing values
print("Missing CPI values:")
print(cpi.isna().sum())

print("\nMissing FX values:")
print(fx.isna().sum())

Missing CPI values:
cpi    0
dtype: int64

Missing FX values:
ngn_usd    0
dtype: int64


In [46]:
#Summary Table
tidy_summary = pd.DataFrame({"Series": ["Nigeria CPI", "Nigeria FX"],
                             "Start": [cpi.index.min(), fx.index.min()],
                             "End": [cpi.index.max(), fx.index.max()],
                             "Observations": [len(cpi), len(fx)],
                             "Frequency": [cpi.index.freqstr, fx.index.freqstr],
                             "Duplicate timestamps": [cpi.index.duplicated().sum(), fx.index.duplicated().sum()],
                             "Missing values": [cpi.isna().sum().iloc[0], fx.isna().sum().iloc[0]]})

display(tidy_summary)

,Series,Start,End,Observations,Frequency,Duplicate timestamps,Missing values
0,Nigeria CPI,2014-01-01,2025-12-01,144,MS,0,0
1,Nigeria FX,2015-01-01,2026-07-01,139,MS,0,0
